# Model Validation — Precision / Recall Sweep

This notebook measures **how well a classification model performs** on recordings you have
already annotated. It reads audio and expert annotations from your **Google Drive**, runs the
model **once**, and then sweeps the detection settings to draw **precision and recall curves**.

---

### Why a sweep?

A model does not have one accuracy — it has one accuracy *per operating point*. Raise the
detection threshold and precision goes up while recall goes down. The point of this notebook is
to show you that whole trade-off curve so you can pick the threshold your project actually needs.

The model is run **once** and its raw outputs (**logits**) are kept. Every threshold and every
sigmoid bias is then applied to those stored logits, which costs milliseconds instead of re-running
inference. That is what makes sweeping dozens of operating points practical.

### What this notebook does (in order):
1. **Connects to your Google Drive** to read audio, annotations, and save results
2. **Installs the necessary software** automatically
3. **Scans your audio folder and your annotations** and pairs them up
4. **Loads a model** from HuggingFace or your Google Drive (e.g. BirdNET, Perch, custom)
5. **Runs the model once per recording** and stores the raw logits
6. **Sweeps thresholds × sigmoid biases**, comparing detections against the annotations
7. **Plots precision and recall curves** — one figure per label, one coloured pair of curves per bias

### Before you start:
- You need a **Google account** with Google Drive
- You need a **TFLite (`.tflite`) or ONNX (`.onnx`) model file** and a matching **labels file**
- You need **annotated recordings** — Raven selection tables, Audacity label tracks, or a single CSV
- Audio and annotations must be on your Google Drive

### How to run:
Run each cell **one at a time**, from top to bottom. Click the ▶ button on the left of each cell,
or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` on the left has not been run. After running, it shows
> `[1]`, `[2]`, etc. If you see an error (red text), read the message — it usually tells you
> exactly what to fix.

---

Created by [biodiversica](https://biodiversica.xyz). For issues, questions, or feedback, open an
issue on [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) or visit
[biodiversica.xyz](https://biodiversica.xyz).

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click
the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

#@markdown **Compute device** — where the model will run during inference.
#@markdown - **CPU**: works on any Colab runtime (slower).
#@markdown - **GPU**: much faster for **ONNX** models, but you must first set the runtime to a GPU
#@markdown   (*Runtime → Change runtime type → T4 GPU*, then re-run from the top).
#@markdown   TFLite models always run on CPU regardless of this setting.
COMPUTE_DEVICE = 'CPU'  #@param ["CPU", "GPU"]

!pip install ai-edge-litert librosa soundfile huggingface_hub -q

# Install the matching ONNX Runtime build. onnxruntime (CPU) and
# onnxruntime-gpu cannot coexist, so we remove one before installing the other.
if COMPUTE_DEVICE == 'GPU':
    !pip uninstall -y onnxruntime onnxruntime-gpu -q
    # Colab ships CUDA 12, but the newest onnxruntime-gpu wheels are built for
    # CUDA 13 (they look for libcudart.so.13). Pin to the last CUDA 12 build.
    !pip install "onnxruntime-gpu==1.22.0" -q
else:
    !pip uninstall -y onnxruntime-gpu -q
    !pip install onnxruntime -q

print(f'\nAll packages installed successfully (compute device: {COMPUTE_DEVICE}).')

---
## Step 2 — Configuration

Fill in the forms below. **You do not need to edit any code** — just type or select your values in
each form and run the cell.

Run all the forms from top to bottom:
1. **General Settings** — where results are written, logits cache
2. **Audio Preprocessing** — optional filtering / speed change
3. **Audio Input** — where your annotated recordings are
4. **Annotations** — where your ground truth is, how it is formatted, **which labels to
   validate**, and how to **translate** label names between your dataset and the model
5. **Model** — where your model is stored and how it works
6. **Sweep & Validation** — the thresholds and biases to test, and how a detection is matched to an annotation

> **Tip:** The forms hide the code automatically. To see the underlying code, click the `{ }` icon
> in the top-right corner of any form cell.

In [ ]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Results folder** — where the validation metrics and figures will be saved on your Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/validation_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Cache model logits on Drive** — store each recording's raw model outputs so that
#@markdown re-running the sweep (with different thresholds or biases) does not re-run the model.
#@markdown The cache is invalidated automatically if the model, the audio settings, or the
#@markdown evaluated label set change.
USE_LOGITS_CACHE = True  #@param {type:"boolean"}

#@markdown **Cache folder** — only used when the cache is enabled.
DRIVE_LOGITS_CACHE_DIR = "/content/drive/MyDrive/validation_results/logits_cache"  #@param {type:"string"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
if USE_LOGITS_CACHE:
    os.makedirs(DRIVE_LOGITS_CACHE_DIR, exist_ok=True)

print(f"Results folder : {DRIVE_RESULTS_DIR}")
print(f"Logits cache   : {DRIVE_LOGITS_CACHE_DIR if USE_LOGITS_CACHE else 'disabled'}")

In [ ]:
#@title ⚙️ Audio Preprocessing { display-mode: "form" }
#@markdown Optional preprocessing applied to each recording before inference.
#@markdown Use the **same settings you intend to use in production** — otherwise the numbers
#@markdown this notebook produces will not describe your real pipeline.
#@markdown
#@markdown ---
#@markdown **Frequency filter** — remove frequencies outside the range of interest.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Low-cut frequency (Hz)** — used for highpass and bandpass filters.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **High-cut frequency (Hz)** — used for lowpass and bandpass filters.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Playback speed** — 1.0 = normal. Below 1.0 slows down and lengthens audio;
#@markdown above 1.0 speeds up and shortens it. Useful for shifting frequency content
#@markdown into the model's expected range (e.g. 0.5x halves all frequencies).
#@markdown Annotation times are always interpreted in the **original** recording's timeline —
#@markdown the notebook converts window times back before comparing.
AUDIO_SPEED = 1.0  #@param {type:"number"}
AUDIO_SPEED = min(max(AUDIO_SPEED, 0.25), 4.0)  # keep within 0.25–4.0

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filter   : lowpass <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filter   : highpass >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filter   : bandpass {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Speed    : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Preprocessing enabled:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Preprocessing : none')

In [ ]:
#@title 🗂️ Audio Input { display-mode: "form" }

#@markdown **Audio folder** — path to the folder on your Google Drive that contains the
#@markdown **annotated** recordings. Subfolders are searched too, and a recording is identified
#@markdown by its path **under this folder** — so two recordings in different subfolders may
#@markdown safely share a file name.
#@markdown Example: `/content/drive/MyDrive/my_project/validation_audio`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

if not os.path.isdir(DRIVE_INPUT_DIR):
    print(f"WARNING: Folder not found: {DRIVE_INPUT_DIR}")
    print("Check the path above — make sure Google Drive is connected and the folder exists.")
else:
    _found = [os.path.join(root, name)
              for root, _, names in os.walk(DRIVE_INPUT_DIR) for name in names
              if name.lower().endswith(('.wav', '.flac', '.mp3'))]
    print(f"Audio folder : {DRIVE_INPUT_DIR}")
    print(f"Audio files  : {len(_found)}")
    for _path in sorted(_found)[:5]:
        print(f"  {os.path.relpath(_path, DRIVE_INPUT_DIR)}")
    if len(_found) > 5:
        print(f"  ... and {len(_found) - 5} more")

In [ ]:
#@title 📝 Annotations { display-mode: "form" }

#@markdown **Annotation format** — how your ground truth is stored.
#@markdown - `raven` — one **Raven Pro selection table** per recording (tab-separated, with a header row).
#@markdown - `audacity` — one **Audacity label track** per recording: `start<TAB>end<TAB>label`, no header.
#@markdown - `csv` — a **single table for the whole dataset**, one row per annotation, with a column
#@markdown   naming the audio file.
ANNOTATION_FORMAT = "raven"  #@param ["raven", "audacity", "csv"]

#@markdown ---
#@markdown **Annotation folder** (`raven` / `audacity`) — the per-recording annotation files.
#@markdown A file is matched to a recording when its name **starts with the audio file's name**
#@markdown (without extension), e.g. `20250615_203000.wav` ↔ `20250615_203000.Table.1.selections.txt`.
DRIVE_ANNOTATION_DIR = "/content/drive/MyDrive/annotations"  #@param {type:"string"}

#@markdown **Annotation table** (`csv` only) — path to the single annotation file (`.csv` or `.txt`).
DRIVE_ANNOTATION_CSV = "/content/drive/MyDrive/annotations/annotations.csv"  #@param {type:"string"}

#@markdown ---
#@markdown ### Column names (used by `raven` and `csv`, ignored by `audacity`)
#@markdown Defaults match a standard Raven Pro selection table.
ANN_START_COLUMN = "Begin Time (s)"  #@param {type:"string"}
ANN_END_COLUMN   = "End Time (s)"    #@param {type:"string"}
ANN_LABEL_COLUMN = "Annotation"      #@param {type:"string"}
#@markdown **File column** (`csv` only) — the column holding the audio file name.
ANN_FILE_COLUMN  = "filename"        #@param {type:"string"}

#@markdown ---
#@markdown ### 🏷️ Label translation
#@markdown Your annotations and your model rarely spell the same label the same way. These two
#@markdown fields rename each side into **one shared vocabulary** — translate whichever side is
#@markdown easier, or both. Format: `from=to` pairs separated by semicolons.
#@markdown
#@markdown Run **Step 3** to see the label names in your annotations, and **Step 4** to have the
#@markdown notebook point out which of them do not match any model label, with suggestions.

#@markdown **Translate annotation labels** — rename a label used in your dataset.
#@markdown Example: `PHYLUT=Phyllodytes luteolus;DENMIN=Dendropsophus minutus`
TRANSLATE_ANNOTATION_LABELS = ""  #@param {type:"string"}

#@markdown **Translate model labels** — rename a label produced by the model.
#@markdown Example: `Phyllodytes luteolus_Bromeliad Treefrog=Phyllodytes luteolus`
#@markdown When several model labels translate to the same name, the highest-scoring one is used.
TRANSLATE_MODEL_LABELS = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### 🎯 Labels to validate
#@markdown **Labels to validate** — semicolon-separated. Leave **blank to validate every label
#@markdown found in your annotations** (the default). Run Step 3 first to see that list, then come
#@markdown back and narrow it down if you only care about some labels.
#@markdown
#@markdown Names are matched **after** the translations above. You may also list a label that never
#@markdown appears in your annotations: it is then scored for **false positives only**, which is how
#@markdown you charge the model for detecting a label that is not there.
#@markdown Example: `Phyllodytes luteolus;Dendropsophus minutus`
VALIDATION_LABELS = ""  #@param {type:"string"}

#@markdown **Ignore these annotation labels** — semicolon-separated. Rows carrying one of these
#@markdown labels are dropped before anything else, so they never reach the list above.
#@markdown Example: `Unknown;Noise;?`
IGNORE_ANNOTATION_LABELS = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Include recordings with no annotation file** — when enabled, such recordings are
#@markdown treated as **fully negative** (every detection in them is a false positive).
#@markdown Only enable this if your annotators really did listen to those files and found nothing.
INCLUDE_UNANNOTATED_FILES = False  #@param {type:"boolean"}


def parse_pair_map(text):
    """Parse a 'from=to;from=to' form field into a dict."""
    mapping = {}
    for entry in (text or '').split(';'):
        entry = entry.strip()
        if '=' not in entry:
            continue
        source, target = entry.split('=', 1)
        source, target = source.strip(), target.strip()
        if source and target:
            mapping[source] = target
    return mapping


def parse_list(text):
    """Parse a 'a;b;c' form field into a list of non-empty strings."""
    return [item.strip() for item in (text or '').split(';') if item.strip()]


ANNOTATION_LABEL_MAP_D = parse_pair_map(TRANSLATE_ANNOTATION_LABELS)
MODEL_LABEL_MAP_D      = parse_pair_map(TRANSLATE_MODEL_LABELS)
IGNORED_LABELS         = set(parse_list(IGNORE_ANNOTATION_LABELS))
SELECTED_LABELS        = parse_list(VALIDATION_LABELS)

print(f"Format            : {ANNOTATION_FORMAT}")
if ANNOTATION_FORMAT == 'csv':
    print(f"Annotation table  : {DRIVE_ANNOTATION_CSV}")
    print(f"Columns           : file='{ANN_FILE_COLUMN}'  start='{ANN_START_COLUMN}'  "
          f"end='{ANN_END_COLUMN}'  label='{ANN_LABEL_COLUMN}'")
else:
    print(f"Annotation folder : {DRIVE_ANNOTATION_DIR}")
    if ANNOTATION_FORMAT == 'raven':
        print(f"Columns           : start='{ANN_START_COLUMN}'  end='{ANN_END_COLUMN}'  "
              f"label='{ANN_LABEL_COLUMN}'")
print(f"Ignored labels    : {sorted(IGNORED_LABELS) or 'none'}")
print(f"Unannotated files : {'treated as fully negative' if INCLUDE_UNANNOTATED_FILES else 'skipped'}")
print()
if ANNOTATION_LABEL_MAP_D or MODEL_LABEL_MAP_D:
    print("Label translation:")
    for source, target in sorted(ANNOTATION_LABEL_MAP_D.items()):
        print(f"  annotation  {source!r} → {target!r}")
    for source, target in sorted(MODEL_LABEL_MAP_D.items()):
        print(f"  model       {source!r} → {target!r}")
else:
    print("Label translation : none — annotation and model labels are compared as they are.")
print()
if SELECTED_LABELS:
    print(f"Labels to validate ({len(SELECTED_LABELS)}):")
    for label in SELECTED_LABELS:
        print(f"  {label}")
else:
    print("Labels to validate : every label found in the annotations (Step 3 lists them).")

In [ ]:
#@title 🤖 Model { display-mode: "form" }

#@markdown **Model access** — where is your model file stored?
MODEL_SOURCE = "google_drive"  #@param ["huggingface", "google_drive"]

#@markdown ---
#@markdown Full path to the model file on your Drive (`.tflite` or `.onnx`).
DRIVE_MODEL_PATH  = "/content/drive/MyDrive/Models/model.tflite"  #@param {type:"string"}
#@markdown Full path to the labels file on your Drive.
DRIVE_LABELS_PATH = "/content/drive/MyDrive/Models/labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown The repo ID is the part after `huggingface.co/` in the model URL.
#@markdown Default: `justinchuby/BirdNET-onnx` (BirdNET v2.4 in ONNX format)
HF_REPO_ID     = "justinchuby/BirdNET-onnx"  #@param {type:"string"}
HF_MODEL_FILE  = "model.onnx"               #@param {type:"string"}
#@markdown The labels file can be in a **different** HuggingFace repo. Leave blank to use the same repo as the model.
HF_LABELS_REPO = ""                          #@param {type:"string"}
HF_LABELS_FILE = "BirdNET_GLOBAL_6K_V2.4_Labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### Labels file settings
#@markdown **Has header row?** — check if the first line of the labels file is a column header (not a label).
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Label column index** — which column contains the label name? (0 = first column, 1 = second, etc.)
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Column delimiter** — how columns are separated in the labels file.
LABELS_DELIMITER = "tab"  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

#@markdown ---
#@markdown **Sigmoid sensitivity** — steepness of the sigmoid curve. `-1.0` = standard sigmoid;
#@markdown more negative = steeper. Must stay **negative**, so that a higher logit always means a
#@markdown higher score. This notebook always activates with a sigmoid — the **bias** is what the
#@markdown sweep varies, so it is set further down.
SIGMOID_SENSITIVITY = -1.0  #@param {type:"number"}
SIGMOID_SENSITIVITY = min(SIGMOID_SENSITIVITY, -0.01)  # keep strictly negative

#@markdown **Sample rate (Hz)** — audio sample rate the model expects.
#@markdown BirdNET = 48000 · Google Perch = 32000 · Custom: check your model documentation.
SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Segment duration (seconds)** — length of each audio chunk fed to the model.
#@markdown BirdNET = 3.0 · Google Perch = 5.0 · Custom: check your model documentation.
SEGMENT_DURATION_S = 3.0  #@param {type:"number"}

#@markdown **Segment overlap (0.0–0.9)** — fraction of overlap between consecutive audio chunks.
SEGMENT_OVERLAP = 0.0  #@param {type:"number"}
SEGMENT_OVERLAP = min(max(SEGMENT_OVERLAP, 0.0), 0.9)  # keep within 0.0–0.9

print(f"Model access        : {MODEL_SOURCE}")
print(f"Activation          : sigmoid (sensitivity={SIGMOID_SENSITIVITY}, bias is swept)")
print(f"Sample rate         : {SAMPLE_RATE} Hz")
print(f"Segment duration    : {SEGMENT_DURATION_S} s")
print(f"Segment overlap     : {SEGMENT_OVERLAP}")
print(f"Labels: column index={LABELS_COLUMN_INDEX}, delimiter='{LABELS_DELIMITER}', header={LABELS_HAS_HEADER}")

In [ ]:
#@title 🎚️ Sweep & Validation { display-mode: "form" }

#@markdown ### Operating points to test
#@markdown **Score thresholds** — comma-separated. This is the **x axis** of the final plots:
#@markdown a detection is kept when its activated score is at or above the threshold.
SCORE_THRESHOLDS = "0.05,0.1,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95"  #@param {type:"string"}

#@markdown **Sigmoid biases** — comma-separated. Each value becomes **one coloured pair of curves**
#@markdown (precision and recall) in every figure. `1.0` = standard sigmoid; above 1.0 shifts the
#@markdown curve left (higher scores, more sensitive); below 1.0 shifts it right (more conservative).
SIGMOID_BIASES = "0.75,1.0,1.25"  #@param {type:"string"}

#@markdown ---
#@markdown ### How a detection is matched to an annotation
#@markdown **Match mode**
#@markdown - `overlap` — any temporal intersection counts as a match (default).
#@markdown - `iou` — requires intersection-over-union at or above the threshold below.
MATCH_MODE = "overlap"  #@param ["overlap", "iou"]
#@markdown **IoU threshold** — only used when match mode is `iou`. Must be between 0 and 1.
IOU_THRESHOLD = 0.5  #@param {type:"number"}

#@markdown ---
#@markdown **Granularity** — what one TP / FP / FN counts.
#@markdown - `window` — one count per **model analysis window** × label. A long annotation spanning
#@markdown   many windows produces many TPs, so results are weighted by annotation duration.
#@markdown - `annotation` — one count per **annotation event**. A burst of detections inside one
#@markdown   annotated call collapses into a single TP, so recall is not inflated by call length.
#@markdown   Detections matching no annotation are still counted as FP. **Recommended.**
#@markdown - `file` — one count per **recording** × label: was the label present anywhere in the
#@markdown   file, and did the model find it anywhere? Times are ignored.
GRANULARITY = "annotation"  #@param ["annotation", "window", "file"]

#@markdown ---
#@markdown **Report true negatives** — only meaningful for `file` granularity, where a TN is a
#@markdown recording in which a label was neither annotated nor detected. TNs do not affect
#@markdown precision or recall.
REPORT_TN = False  #@param {type:"boolean"}


def parse_float_list(text, lo, hi):
    """Parse a comma-separated form field into sorted, unique, clamped floats."""
    values = []
    for item in (text or '').replace(';', ',').split(','):
        item = item.strip()
        if not item:
            continue
        try:
            values.append(min(max(float(item), lo), hi))
        except ValueError:
            print(f"  WARNING: could not read '{item}' as a number — skipping it.")
    return sorted(set(values))


THRESHOLDS = parse_float_list(SCORE_THRESHOLDS, 0.0, 1.0)
BIASES     = parse_float_list(SIGMOID_BIASES, 0.01, 1.99)

if not THRESHOLDS:
    raise ValueError("No usable score thresholds. Enter at least one number between 0 and 1.")
if not BIASES:
    raise ValueError("No usable sigmoid biases. Enter at least one number between 0.01 and 1.99.")
if MATCH_MODE == 'iou' and not (0 < IOU_THRESHOLD < 1):
    raise ValueError(f"IOU_THRESHOLD must be between 0 and 1 (exclusive), got {IOU_THRESHOLD}.")

print(f"Score thresholds : {THRESHOLDS}")
print(f"Sigmoid biases   : {BIASES}")
print(f"Operating points : {len(THRESHOLDS) * len(BIASES)}  "
      f"({len(BIASES)} curve pair(s) × {len(THRESHOLDS)} point(s))")
print(f"Match mode       : {MATCH_MODE}" + (f" (IoU >= {IOU_THRESHOLD})" if MATCH_MODE == 'iou' else ""))
print(f"Granularity      : {GRANULARITY}")

# With the standard sigmoid, score >= t is the same cut as
# logit >= ln(t / (1 - t)) - 10 * (bias - 1): the threshold and the bias move
# the same operating point along one axis. Sweeping both therefore re-measures
# points you already have — the bias curves are horizontal shifts of each other.
# That is exactly what makes them comparable, but it is worth knowing that a
# bias curve is not new information about the model, only a re-labelled x axis.
if len(BIASES) > 1:
    print()
    print("Note: with a sigmoid, score threshold and sigmoid bias move the same operating point.")
    print("      The bias curves are horizontal shifts of one another — useful for reading off")
    print("      the threshold you would need at each bias, not extra information about the model.")

---
## Step 3 — Scan audio files and load annotations

This cell finds your recordings, reads the annotations, pairs them up, and reports what will be
evaluated.

**How a recording is matched to its annotations** — a recording is identified by its path *under
the audio folder*, not by its file name alone, so `POINT_A/20250615_203000.wav` and
`POINT_B/20250615_203000.wav` are two different recordings:

- `raven` / `audacity` — the annotation file whose name starts with the recording's file name.
  When the audio folder has subfolders, an annotation file **in the matching subfolder** wins; if
  the annotation folder is flat, a file name unique across the whole folder is used.
- `csv` — a row's file column may hold either the full path under the audio folder
  (`POINT_A/20250615_203000.wav`) or just the file name, as long as that name is unique in the table.

If a recording's file name is reused in several subfolders and nothing distinguishes them, the cell
**skips that recording and says so** rather than merging two sets of ground truth.

**What to check in the output:**
- **Recordings to validate** — if this is 0, your annotation files are not being matched. Check
  that the annotation file names start with the audio file names.
- **Labels found in the annotations** — the full list of label names your dataset uses, with a
  count each. This is the menu: leave `VALIDATION_LABELS` blank to validate all of them, or copy
  the ones you care about into that field and re-run this cell.
- **Label translation warnings** — a translation whose left-hand side never matched any annotation
  is reported here, since a typo there looks exactly like a missing label.

Step 4 then checks each label against the model's own vocabulary and suggests translations for any
that do not line up.

In [ ]:
#@title 🔍 Scan { display-mode: "form" }

import csv
import difflib
import glob as _glob
from collections import defaultdict

AUDIO_EXTENSIONS = ('.wav', '.flac', '.mp3')


def close_names(name, candidates, n=3, cutoff=0.6):
    """Nearest candidate names to `name`, compared without case.

    A label that differs only in capitalisation is the most common reason two
    vocabularies fail to line up, and difflib scores that pair at zero — so the
    comparison is done lowercased and the original spelling is handed back.
    """
    by_lower = {candidate.lower(): candidate for candidate in candidates}
    return [by_lower[match] for match in
            difflib.get_close_matches(name.lower(), list(by_lower), n=n, cutoff=cutoff)]


def recording_key(path, root):
    """A recording's identity: its path under `root`, without the extension.

    Two recordings in different subfolders can share a file name, so the
    subfolder has to be part of the key — keying on the file name alone would
    attach one subfolder's annotations to another's audio.
    """
    return os.path.splitext(os.path.relpath(path, root))[0].replace(os.sep, '/')


def name_matches(basename, stem):
    """Does an annotation file name belong to the recording with this stem?"""
    return (basename == stem
            or basename.startswith(stem + '.')
            or basename.startswith(stem + '_'))


def sniff_delimiter(sample_line):
    """Pick the delimiter of a text table: tab wins, then semicolon, then comma."""
    for delimiter in ('\t', ';', ','):
        if delimiter in sample_line:
            return delimiter
    return ','


def read_table(path):
    """Read a delimited text table with a header row into a list of dicts."""
    with open(path, 'r', encoding='utf-8-sig', newline='') as handle:
        first = handle.readline()
        if not first:
            return []
        handle.seek(0)
        return list(csv.DictReader(handle, delimiter=sniff_delimiter(first)))


# Every label as written in the files, before translation. Kept so the scan can
# report a translation whose left-hand side never matched anything — a silent
# typo that would otherwise just look like a missing label.
raw_label_counts = defaultdict(int)


def clean_annotation(start, end, label):
    """Normalise one raw annotation, or return None if it cannot be used."""
    try:
        start, end = float(start), float(end)
    except (TypeError, ValueError):
        return None
    if end <= start:
        return None
    label = (label or '').strip()
    if not label:
        return None
    raw_label_counts[label] += 1
    if label in IGNORED_LABELS:
        return None
    # Translation is applied here, once, so everything downstream sees one vocabulary.
    return {'start_time': start, 'end_time': end,
            'label': ANNOTATION_LABEL_MAP_D.get(label, label)}


def read_raven_table(path):
    """Read a Raven Pro selection table."""
    rows = read_table(path)
    if not rows:
        return []
    columns = rows[0].keys()
    for required in (ANN_START_COLUMN, ANN_END_COLUMN, ANN_LABEL_COLUMN):
        if required not in columns:
            raise KeyError(
                f"Column '{required}' not found in {os.path.basename(path)}.\n"
                f"Columns present: {list(columns)}\n"
                "Fix the column names in the Annotations form (Step 2)."
            )
    annotations = []
    for row in rows:
        # Raven writes one row per view when both Waveform and Spectrogram are
        # exported; keeping both would double every count.
        view = (row.get('View') or '').strip()
        if view and not view.lower().startswith('spectrogram'):
            continue
        annotation = clean_annotation(row.get(ANN_START_COLUMN), row.get(ANN_END_COLUMN),
                                      row.get(ANN_LABEL_COLUMN))
        if annotation:
            annotations.append(annotation)
    return annotations


def read_audacity_labels(path):
    """Read an Audacity label track: start<TAB>end<TAB>label, no header."""
    annotations = []
    with open(path, 'r', encoding='utf-8-sig') as handle:
        for line in handle:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            annotation = clean_annotation(parts[0], parts[1], parts[2])
            if annotation:
                annotations.append(annotation)
    return annotations


def read_annotation_csv(path):
    """Read one table covering the whole dataset, keyed by audio file name.

    Returns (by_key, aliases). `by_key` is keyed on the file column with the
    extension stripped, exactly as written — so `POINT_A/rec.wav` becomes
    `POINT_A/rec`. `aliases` maps a bare file name to the single key that owns
    it, or to None when several keys share it: a name used by two different
    recordings cannot be resolved, and guessing would silently mix up their
    ground truth.
    """
    rows = read_table(path)
    if not rows:
        return {}, {}
    columns = rows[0].keys()
    for required in (ANN_FILE_COLUMN, ANN_START_COLUMN, ANN_END_COLUMN, ANN_LABEL_COLUMN):
        if required not in columns:
            raise KeyError(
                f"Column '{required}' not found in {os.path.basename(path)}.\n"
                f"Columns present: {list(columns)}\n"
                "Fix the column names in the Annotations form (Step 2)."
            )
    by_key = {}
    for row in rows:
        raw = (row.get(ANN_FILE_COLUMN) or '').strip().replace('\\', '/').lstrip('./')
        key = os.path.splitext(raw)[0]
        if not key:
            continue
        # Register the recording even when every one of its rows is filtered
        # out: it was still reviewed, so it is an all-negative recording rather
        # than one with missing annotations.
        by_key.setdefault(key, [])
        annotation = clean_annotation(row.get(ANN_START_COLUMN), row.get(ANN_END_COLUMN),
                                      row.get(ANN_LABEL_COLUMN))
        if annotation:
            by_key[key].append(annotation)

    aliases = {}
    for key in by_key:
        name = key.rsplit('/', 1)[-1]
        aliases[name] = None if name in aliases else key
    return by_key, aliases


# --- find the audio -----------------------------------------------------------
if not os.path.isdir(DRIVE_INPUT_DIR):
    raise FileNotFoundError(
        f"Audio folder not found: {DRIVE_INPUT_DIR}\n"
        "Please check the path in the Audio Input form (Step 2)."
    )

all_audio = sorted(
    path for extension in AUDIO_EXTENSIONS
    for path in _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', f'*{extension}'), recursive=True)
)

if not all_audio:
    raise FileNotFoundError(
        f"No audio files found in: {DRIVE_INPUT_DIR}\n"
        f"Supported formats: {', '.join(AUDIO_EXTENSIONS)}"
    )

# --- read the annotations -----------------------------------------------------
annotations_by_key = {}
missing_annotations = []
ambiguous_annotations = []

if ANNOTATION_FORMAT == 'csv':
    if not os.path.exists(DRIVE_ANNOTATION_CSV):
        raise FileNotFoundError(
            f"Annotation table not found: {DRIVE_ANNOTATION_CSV}\n"
            "Please check the path in the Annotations form (Step 2)."
        )
    csv_annotations, csv_aliases = read_annotation_csv(DRIVE_ANNOTATION_CSV)
    for audio_path in all_audio:
        key = recording_key(audio_path, DRIVE_INPUT_DIR)
        name = key.rsplit('/', 1)[-1]
        if key in csv_annotations:
            annotations_by_key[key] = csv_annotations[key]
        elif csv_aliases.get(name):
            # The table named the recording by file name alone, and only one
            # recording in it goes by that name.
            annotations_by_key[key] = csv_annotations[csv_aliases[name]]
        elif name in csv_aliases:
            ambiguous_annotations.append((key, f"'{name}' names several recordings in the table"))
else:
    if not os.path.isdir(DRIVE_ANNOTATION_DIR):
        raise FileNotFoundError(
            f"Annotation folder not found: {DRIVE_ANNOTATION_DIR}\n"
            "Please check the path in the Annotations form (Step 2)."
        )
    annotation_files = sorted(
        path for path in _glob.glob(os.path.join(DRIVE_ANNOTATION_DIR, '**', '*'), recursive=True)
        if os.path.isfile(path) and not path.lower().endswith(AUDIO_EXTENSIONS)
    )
    reader = read_raven_table if ANNOTATION_FORMAT == 'raven' else read_audacity_labels
    for audio_path in all_audio:
        stem = os.path.splitext(os.path.basename(audio_path))[0]
        relative_dir = os.path.dirname(os.path.relpath(audio_path, DRIVE_INPUT_DIR))
        matches = [path for path in annotation_files
                   if name_matches(os.path.basename(path), stem)]
        # Prefer annotation files sitting in the same subfolder as the recording,
        # so a file name reused across subfolders resolves to its own.
        same_folder = [path for path in matches
                       if os.path.dirname(os.path.relpath(path, DRIVE_ANNOTATION_DIR))
                       == relative_dir]
        chosen = same_folder or matches
        if not chosen:
            continue
        key = recording_key(audio_path, DRIVE_INPUT_DIR)
        if not same_folder and len({os.path.dirname(path) for path in chosen}) > 1:
            # Same file name in several annotation folders and none of them
            # mirrors the audio's folder: merging them would invent annotations.
            ambiguous_annotations.append(
                (key, 'matches ' + ', '.join(os.path.relpath(p, DRIVE_ANNOTATION_DIR)
                                             for p in chosen)))
            continue
        found = []
        for path in chosen:
            found.extend(reader(path))
        annotations_by_key[key] = found

# --- pair audio with annotations ---------------------------------------------
validation_files = []
for audio_path in all_audio:
    key = recording_key(audio_path, DRIVE_INPUT_DIR)
    if key in annotations_by_key:
        found = annotations_by_key[key]
    elif INCLUDE_UNANNOTATED_FILES and not any(k == key for k, _ in ambiguous_annotations):
        found = []
    else:
        if not any(k == key for k, _ in ambiguous_annotations):
            missing_annotations.append(key)
        continue
    validation_files.append({'path': audio_path, 'key': key, 'annotations': found})

if not validation_files:
    raise FileNotFoundError(
        "No recording could be paired with annotations.\n"
        f"Audio files found: {len(all_audio)}  |  Annotation sets read: {len(annotations_by_key)}\n"
        + (
            f"{len(ambiguous_annotations)} recording(s) matched several annotation sets and none "
            "could be resolved — e.g. "
            f"'{ambiguous_annotations[0][0]}' {ambiguous_annotations[0][1]}.\n"
            "Mirror the audio folder's subfolders in the annotation folder, or name the "
            "recordings by their full relative path."
            if ambiguous_annotations else
            "Check that annotation file names start with the audio file name (without "
            "extension), or switch INCLUDE_UNANNOTATED_FILES on if the recordings really "
            "are all-negative."
        )
    )

# --- the labels to validate ---------------------------------------------------
annotation_label_counts = defaultdict(int)
for entry in validation_files:
    for annotation in entry['annotations']:
        annotation_label_counts[annotation['label']] += 1

AVAILABLE_LABELS = sorted(annotation_label_counts)
if not AVAILABLE_LABELS and not SELECTED_LABELS:
    raise ValueError(
        "No labels to validate: the annotations are empty after filtering.\n"
        "Check IGNORE_ANNOTATION_LABELS and the label column name in the Annotations form."
    )

# Blank selection means "everything the dataset has"; an explicit selection wins,
# and may name a label the dataset never used (scored for false positives only).
EVAL_LABELS = sorted(set(SELECTED_LABELS)) if SELECTED_LABELS else AVAILABLE_LABELS

total_annotations = sum(annotation_label_counts.values())
print(f"Audio folder            : {DRIVE_INPUT_DIR}")
print(f"Audio files found       : {len(all_audio)}")
print(f"Recordings to validate  : {len(validation_files)}")
print(f"Annotations read        : {total_annotations}")
print()

print(f"Labels found in the annotations ({len(AVAILABLE_LABELS)}):")
for label in AVAILABLE_LABELS:
    note = '' if label in EVAL_LABELS else '   ← not selected, skipped'
    print(f"  {label:<45} {annotation_label_counts[label]:>6} annotation(s){note}")

print()
if SELECTED_LABELS:
    skipped = [label for label in AVAILABLE_LABELS if label not in EVAL_LABELS]
    print(f"Validating {len(EVAL_LABELS)} selected label(s):")
    for label in EVAL_LABELS:
        count = annotation_label_counts.get(label, 0)
        note  = f'{count:>6} annotation(s)' if count else '     false positives only (no annotations)'
        print(f"  {label:<45} {note}")
    if skipped:
        print(f"  ({len(skipped)} annotated label(s) left out of this run: {', '.join(skipped[:6])}"
              f"{' ...' if len(skipped) > 6 else ''})")
    # A selected label with no annotations that closely resembles one that has
    # them is nearly always a typo or a translation that has not been set up.
    for label in EVAL_LABELS:
        if annotation_label_counts.get(label):
            continue
        close = close_names(label, AVAILABLE_LABELS, cutoff=0.75)
        if close:
            print(f"  NOTE: '{label}' matched no annotation but resembles {close}.")
            print(f"        Either fix the spelling in VALIDATION_LABELS, or add a translation")
            print(f"        like '{close[0]}={label}' to TRANSLATE_ANNOTATION_LABELS.")
else:
    print("Validating every label above.")
    print("To validate only some of them, list them in VALIDATION_LABELS (Annotations form, Step 2).")

unused_translations = sorted(set(ANNOTATION_LABEL_MAP_D) - set(raw_label_counts))
if unused_translations:
    print()
    print("WARNING: these annotation translations never matched any annotation label:")
    for source in unused_translations:
        close = close_names(source, sorted(raw_label_counts), n=2)
        print(f"  '{source}'" + (f"   — did you mean {close}?" if close else ''))

if missing_annotations:
    print()
    print(f"WARNING: {len(missing_annotations)} recording(s) had no annotation file and were skipped:")
    for name in missing_annotations[:5]:
        print(f"  {name}")
    if len(missing_annotations) > 5:
        print(f"  ... and {len(missing_annotations) - 5} more")

if ambiguous_annotations:
    print()
    print(f"WARNING: {len(ambiguous_annotations)} recording(s) could not be matched to one "
          f"annotation set and were skipped:")
    for key, reason in ambiguous_annotations[:5]:
        print(f"  {key}  —  {reason}")
    if len(ambiguous_annotations) > 5:
        print(f"  ... and {len(ambiguous_annotations) - 5} more")
    print("  Mirror the audio folder's subfolders in the annotation folder, or name the")
    print("  recordings by their full relative path, so each one resolves to its own set.")

print()
print("Scan complete. Continue to Step 4.")

---
## Step 4 — Load the model and its labels

This cell loads your model and reads the list of classes it can detect, then checks that the labels
you are validating actually exist in the model.

**Labels file format:** a plain text file with one label per line, for example:
```
Phyllodytes luteolus
Dendropsophus minutus
Background noise
```

The number of lines in the labels file must match the number of outputs of your model.

In [ ]:
#@title 🧠 Load model and labels { display-mode: "form" }

import numpy as np

_DELIMITERS = {"tab": "\t", "comma (,)": ",", "semicolon (;)": ";", "underscore (_)": "_"}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Downloading model from HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
    _labels_repo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'Downloading labels from HuggingFace: {_labels_repo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_labels_repo, filename=HF_LABELS_FILE)
elif MODEL_SOURCE == 'google_drive':
    model_path  = DRIVE_MODEL_PATH
    labels_path = DRIVE_LABELS_PATH
else:
    raise ValueError(f"MODEL_SOURCE must be 'huggingface' or 'google_drive', got: {MODEL_SOURCE!r}")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}\n"
        "Please check the path in the Model form (Step 2)."
    )
if not os.path.exists(labels_path):
    raise FileNotFoundError(
        f"Labels file not found: {labels_path}\n"
        "Please check the path in the Model form (Step 2)."
    )

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Unsupported model format: '{ext}'.\nThe model file must end in .tflite or .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    model = TFLiteInterpreter(model_path=model_path)
    model.allocate_tensors()
    in_shape  = model.get_input_details()[0]['shape']
    out_shape = model.get_output_details()[0]['shape']
    print('TFLite model loaded.')
    print(f'  Input  shape : {in_shape}')
    print(f'  Output shape : {out_shape}')
    if globals().get('COMPUTE_DEVICE', 'CPU') == 'GPU':
        print('  Note         : TFLite models run on CPU only; the GPU setting is ignored.')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort

    # Select execution provider based on the device chosen in Step 1.
    _device    = globals().get('COMPUTE_DEVICE', 'CPU')
    _available = ort.get_available_providers()
    if _device == 'GPU' and 'CUDAExecutionProvider' in _available:
        _providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    else:
        if _device == 'GPU':
            print('WARNING: GPU requested but CUDAExecutionProvider is not available — falling back to CPU.')
            print('         Set the Colab runtime to a GPU (Runtime → Change runtime type → T4 GPU)')
            print('         and re-run from Step 1 to use the GPU.')
        _providers = ['CPUExecutionProvider']

    model = ort.InferenceSession(model_path, providers=_providers)
    in_shape  = model.get_inputs()[0].shape
    out_shape = model.get_outputs()[0].shape
    print('ONNX model loaded.')
    print(f'  Input  shape : {in_shape}')
    print(f'  Output shape : {out_shape}')
    print(f'  Device       : {_device}  ({model.get_providers()})')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

if LABELS_HAS_HEADER and lines:
    lines = lines[1:]

for line in lines:
    line = line.strip()
    if not line:
        continue
    if _labels_sep in line:
        parts = line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(parts):
            labels.append(parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f"  WARNING: column index {LABELS_COLUMN_INDEX} not found in line: {line!r} — skipping.")
    else:
        labels.append(line)

print(f'\nLabels loaded: {len(labels)} classes')
print(f'  Delimiter : {LABELS_DELIMITER}  |  Column index : {LABELS_COLUMN_INDEX}  |  Header skipped : {LABELS_HAS_HEADER}')
print(f'  First 5   : {labels[:5]}')
print(f'  Model name: {MODEL_NAME}')

# --- map model outputs onto the labels being evaluated ------------------------
# Only these columns of the model's output are kept, which is what makes caching
# every recording's logits cheap even for a 6000-class model. Several model
# labels may map onto one evaluated label; the highest-scoring one wins.
EVAL_COLUMNS = {label: [] for label in EVAL_LABELS}
for index, model_label in enumerate(labels):
    canonical = MODEL_LABEL_MAP_D.get(model_label, model_label)
    if canonical in EVAL_COLUMNS:
        EVAL_COLUMNS[canonical].append(index)

unpredictable = [label for label, columns in EVAL_COLUMNS.items() if not columns]
print()
print(f'Labels to validate matched to model outputs: '
      f'{len(EVAL_LABELS) - len(unpredictable)}/{len(EVAL_LABELS)}')
for label in EVAL_LABELS:
    if EVAL_COLUMNS[label]:
        aliases = [labels[i] for i in EVAL_COLUMNS[label]]
        detail = f'  ← {aliases}' if aliases != [label] else ''
        print(f'  {label:<45} ok{detail}')

if unpredictable:
    # What the model can actually emit, after its own translation.
    model_vocabulary = sorted({MODEL_LABEL_MAP_D.get(name, name) for name in labels})
    print()
    print('WARNING: these labels have no matching model output. The model can never predict')
    print('         them, so their precision stays undefined and their recall stays 0.')
    for label in unpredictable:
        print(f'  {label}')
        close = close_names(label, model_vocabulary)
        if close:
            print(f'      closest model label(s) : {close}')
            # Translating the model side keeps your own spelling in the results;
            # translating the annotation side would rename your labels to the
            # model's, which is rarely what you want in a report.
            print( '      to link them, add this to TRANSLATE_MODEL_LABELS (Step 2):')
            print(f'        {close[0]}={label}')
        else:
            print('      no similar model label found — check that this model really covers '
                  'this label.')

---
## Step 5 — Run the model once and store the logits

This is the only expensive step. For each recording:
1. The audio is copied from Drive, decoded and preprocessed
2. It is split into windows (e.g. 3 seconds each)
3. Every window is fed to the model and its **raw outputs (logits)** are kept — no threshold, no
   activation, nothing thrown away

Because the logits are stored, the sweep in Step 6 costs milliseconds per operating point instead
of a full re-run of the model. If the logits cache is enabled in Step 2, they are also written to
your Drive so that a later session can skip this step entirely.

> Depending on the number and length of your recordings, this step can take a while. Progress is
> shown below.

In [ ]:
#@title 🚀 Extract logits { display-mode: "form" }

#@markdown **Batch size** — windows sent to the model at once. Larger batches help a lot on GPU
#@markdown runtimes with ONNX models. Models without a dynamic batch axis fall back to one window
#@markdown at a time automatically.
BATCH_SIZE = 32  #@param {type:"integer"}
BATCH_SIZE = max(1, int(BATCH_SIZE))

import hashlib
import shutil
import time
import librosa

segment_samples = int(SEGMENT_DURATION_S * SAMPLE_RATE)
stride_samples  = max(1, int(segment_samples * (1 - SEGMENT_OVERLAP)))

# Any change to these invalidates a cached logits file — the numbers inside it
# would no longer describe the run being asked for.
CACHE_SIGNATURE = hashlib.sha1(repr((
    os.path.abspath(model_path), MODEL_NAME, SAMPLE_RATE, SEGMENT_DURATION_S, SEGMENT_OVERLAP,
    FILTER_TYPE, FILTER_LOW_HZ, FILTER_HIGH_HZ, AUDIO_SPEED, tuple(EVAL_LABELS),
    tuple(tuple(EVAL_COLUMNS[label]) for label in EVAL_LABELS),
)).encode()).hexdigest()[:16]


def preprocess_audio(audio, sr):
    if FILTER_TYPE != 'none':
        from scipy.signal import butter, sosfilt
        nyq = sr / 2.0
        if FILTER_TYPE == 'lowpass':
            sos = butter(5, min(FILTER_HIGH_HZ, nyq - 1) / nyq, btype='low', output='sos')
        elif FILTER_TYPE == 'highpass':
            sos = butter(5, max(FILTER_LOW_HZ, 1) / nyq, btype='high', output='sos')
        elif FILTER_TYPE == 'bandpass':
            lo = max(FILTER_LOW_HZ, 1) / nyq
            hi = min(FILTER_HIGH_HZ, nyq - 1) / nyq
            sos = butter(5, [lo, hi], btype='band', output='sos')
        audio = sosfilt(sos, audio).astype(np.float32)
    if AUDIO_SPEED != 1.0:
        audio = librosa.effects.time_stretch(audio, rate=AUDIO_SPEED)
    return audio


_batch_supported = [None]  # probed lazily on the first real batch


def run_model_batch(segments):
    """Run a (n_windows, n_samples) float32 array through the model.

    Returns raw logits as (n_windows, n_outputs). Falls back to one window at a
    time for models without a batch axis, so a fixed-batch TFLite graph still
    works — just more slowly.
    """
    segments = np.ascontiguousarray(segments, dtype=np.float32)

    if MODEL_TYPE == 'onnx' and _batch_supported[0] is not False and len(segments) > 1:
        input_name = model.get_inputs()[0].name
        try:
            out = model.run(None, {input_name: segments})[0]
            _batch_supported[0] = True
            return np.asarray(out, dtype=np.float32).reshape(len(segments), -1)
        except Exception:
            if _batch_supported[0] is None:
                print('  Note: this model has no dynamic batch axis — running one window at a time.')
            _batch_supported[0] = False

    outputs = []
    for segment in segments:
        if MODEL_TYPE == 'tflite':
            in_det  = model.get_input_details()[0]
            out_det = model.get_output_details()[0]
            try:
                model.set_tensor(in_det['index'], segment.reshape(in_det['shape']))
            except ValueError:
                model.set_tensor(in_det['index'], segment.reshape(1, -1))
            model.invoke()
            outputs.append(model.get_tensor(out_det['index']).flatten())
        else:
            input_name = model.get_inputs()[0].name
            try:
                out = model.run(None, {input_name: segment.reshape(1, -1)})[0]
            except Exception:
                out = model.run(None, {input_name: segment[np.newaxis, :]})[0]
            outputs.append(np.asarray(out, dtype=np.float32).flatten())
    return np.stack(outputs).astype(np.float32)


def select_eval_columns(logits):
    """Reduce full model logits to one column per evaluated label.

    Where several model labels map to the same evaluated label, the largest
    logit wins. With a negative sigmoid sensitivity the activation is monotone
    increasing, so the largest logit is also the largest score at every
    threshold and bias in the sweep — the reduction can safely happen here,
    before any activation.
    """
    selected = np.full((len(logits), len(EVAL_LABELS)), -1e9, dtype=np.float32)
    for column, label in enumerate(EVAL_LABELS):
        indices = [i for i in EVAL_COLUMNS[label] if i < logits.shape[1]]
        if indices:
            selected[:, column] = logits[:, indices].max(axis=1)
    return selected


def cache_path_for(entry):
    # The key carries the subfolder, so two recordings that share a file name
    # across subfolders still get their own cache entry.
    safe = ''.join(c if c.isalnum() or c in '._-' else '_' for c in entry['key'])
    return os.path.join(DRIVE_LOGITS_CACHE_DIR, f'{safe}.{MODEL_NAME}.logits.npz')


def load_cached_logits(entry):
    """Return (starts, ends, logits) from the cache, or None if unusable."""
    path = cache_path_for(entry)
    if not os.path.exists(path):
        return None
    try:
        with np.load(path, allow_pickle=False) as data:
            if str(data['signature']) != CACHE_SIGNATURE:
                return None
            return data['starts'], data['ends'], data['logits']
    except Exception as error:
        print(f'  WARNING: could not read cache {os.path.basename(path)} ({error}) — recomputing.')
        return None


LOCAL_AUDIO_TMP = '/content/audio_tmp'
_last_remount = [0.0]


def copy_from_drive(src, dst, retries=3, remount_cooldown=30):
    """Copy with retries; a dead FUSE mount needs a remount, not just a retry."""
    last_error = None
    for attempt in range(retries):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as error:
            last_error = error
            # A FUSE 'Transport endpoint is not connected' error means the mount
            # itself has died and retrying alone will never succeed. The cooldown
            # keeps a run of unreadable files from remounting over and over.
            if time.time() - _last_remount[0] > remount_cooldown:
                print(f'  WARNING: Google Drive read failed ({error}) — remounting and retrying...')
                try:
                    from google.colab import drive
                    drive.mount('/content/drive', force_remount=True)
                except Exception as remount_error:
                    print(f'  WARNING: remount failed: {remount_error}')
                _last_remount[0] = time.time()
            time.sleep(2 * (attempt + 1))
    raise last_error


def extract_logits(entry):
    """Decode one recording, window it, and return (starts, ends, logits)."""
    os.makedirs(LOCAL_AUDIO_TMP, exist_ok=True)
    local_path = os.path.join(LOCAL_AUDIO_TMP, os.path.basename(entry['path']))
    try:
        copy_from_drive(entry['path'], local_path)
        audio, _ = librosa.load(local_path, sr=SAMPLE_RATE, mono=True)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)

    audio = preprocess_audio(audio, SAMPLE_RATE)

    starts, segments = [], []
    for start_sample in range(0, len(audio), stride_samples):
        segment = audio[start_sample:start_sample + segment_samples]
        if len(segment) < segment_samples * 0.5:
            continue
        if len(segment) < segment_samples:
            segment = np.pad(segment, (0, segment_samples - len(segment)))
        starts.append(start_sample / SAMPLE_RATE)
        segments.append(segment.astype(np.float32))

    if not segments:
        return np.zeros(0, np.float32), np.zeros(0, np.float32), \
               np.zeros((0, len(EVAL_LABELS)), np.float32)

    chunks = [run_model_batch(np.stack(segments[i:i + BATCH_SIZE]))
              for i in range(0, len(segments), BATCH_SIZE)]
    logits = select_eval_columns(np.concatenate(chunks, axis=0))

    # Preprocessing may have stretched the audio; annotations are always given
    # in the original recording's timeline, so convert the window bounds back.
    starts = np.asarray(starts, dtype=np.float32) * AUDIO_SPEED
    ends   = starts + np.float32(SEGMENT_DURATION_S * AUDIO_SPEED)
    return starts, ends, logits


print(f'Extracting logits for {len(validation_files)} recording(s) with model "{MODEL_NAME}"')
print(f'Evaluated labels : {len(EVAL_LABELS)}  |  Batch size: {BATCH_SIZE}')
print(f'Window           : {SEGMENT_DURATION_S}s  |  Overlap: {SEGMENT_OVERLAP}  |  {SAMPLE_RATE} Hz')
print(f'Logits cache     : {DRIVE_LOGITS_CACHE_DIR if USE_LOGITS_CACHE else "disabled"}')
print('=' * 70)

n_cached = n_computed = n_failed = 0
total_windows = 0
run_start = time.time()

for index, entry in enumerate(validation_files, 1):
    name = os.path.basename(entry['path'])
    cached = load_cached_logits(entry) if USE_LOGITS_CACHE else None

    if cached is not None:
        entry['starts'], entry['ends'], entry['logits'] = cached
        n_cached += 1
        print(f"[{index}/{len(validation_files)}] {name}  →  {len(entry['starts'])} windows (from cache)")
    else:
        file_start = time.time()
        try:
            entry['starts'], entry['ends'], entry['logits'] = extract_logits(entry)
        except Exception as error:
            print(f'[{index}/{len(validation_files)}] {name}  ERROR: {error} — skipping.')
            entry['logits'] = None
            n_failed += 1
            continue
        n_computed += 1
        elapsed = time.time() - file_start
        print(f"[{index}/{len(validation_files)}] {name}  →  {len(entry['starts'])} windows  "
              f"({elapsed:.1f}s)")
        if USE_LOGITS_CACHE:
            try:
                np.savez_compressed(cache_path_for(entry), starts=entry['starts'],
                                    ends=entry['ends'], logits=entry['logits'],
                                    signature=np.array(CACHE_SIGNATURE))
            except Exception as error:
                print(f'  WARNING: could not write cache: {error}')

    total_windows += len(entry['starts'])

validation_files = [entry for entry in validation_files if entry.get('logits') is not None]
if not validation_files:
    raise RuntimeError('No recording could be analyzed — see the errors above.')

print()
print('=' * 70)
print('Logit extraction complete.')
print(f'  Recordings : {len(validation_files)}  ({n_computed} analyzed, {n_cached} from cache, {n_failed} failed)')
print(f'  Windows    : {total_windows}')
print(f'  Total time : {time.time() - run_start:.1f}s')
print(f'  Memory held: {sum(e["logits"].nbytes for e in validation_files) / 1e6:.1f} MB of logits')

---
## Step 6 — Sweep the operating points and score them

Now the cheap part. For every combination of **score threshold** × **sigmoid bias**:

1. The stored logits are activated into scores: `score = 1 / (1 + exp(sensitivity × (logit + 10 × (bias − 1))))`
2. Every score at or above the threshold becomes a detection
3. Detections are compared against the annotations to produce **TP / FP / FN** counts
4. Counts become **precision**, **recall** and **F1**, per label, plus micro- and macro-averaged
   summary rows

**How the counting works**, depending on the granularity you chose in Step 2:

| Granularity | TP | FP | FN |
|---|---|---|---|
| `annotation` | an annotation with at least one overlapping same-label detection (each detection is claimed by at most one annotation) | a detection falling inside no same-label annotation | an annotation with no unclaimed overlapping detection |
| `window` | a window where the label is both detected and annotated | a window where the label is detected but not annotated | each annotation overlapping a window where the label was not detected |
| `file` | a recording where the label is both detected and annotated | detected but never annotated in that recording | annotated but never detected in that recording |

An undefined metric — precision for a label the model never predicted, recall for a label that was
never annotated — is left **empty** rather than reported as 0, but counts as 0 in the macro average
(the same convention scikit-learn uses).

In [ ]:
#@title 🎚️ Sweep { display-mode: "form" }

MICRO_AVERAGE_LABEL = '__micro_avg__'
MACRO_AVERAGE_LABEL = '__macro_avg__'


def windows_match(window_starts, window_ends, ann_start, ann_end):
    """Boolean mask over windows: which ones match this annotation?"""
    overlap = np.minimum(window_ends, ann_end) - np.maximum(window_starts, ann_start)
    if MATCH_MODE == 'overlap':
        return overlap > 0
    overlap = np.maximum(overlap, 0.0)
    union   = (window_ends - window_starts) + (ann_end - ann_start) - overlap
    with np.errstate(divide='ignore', invalid='ignore'):
        iou = np.where(union > 0, overlap / union, 0.0)
    return iou >= IOU_THRESHOLD


def build_match_index(entry):
    """Precompute everything about annotation ↔ window matching, once per file.

    None of this depends on the threshold or the bias, so it is computed once
    and reused for every operating point in the sweep.
    """
    starts, ends = entry['starts'], entry['ends']
    n_windows    = len(starts)

    # n_ann[w, l] — how many annotations of label l match window w.
    n_ann = np.zeros((n_windows, len(EVAL_LABELS)), dtype=np.int32)
    # per label: for each annotation, the windows it matches (annotation order)
    ann_windows = {label: [] for label in EVAL_LABELS}
    # per label: how many annotations of it exist in this recording at all
    ann_totals = {label: 0 for label in EVAL_LABELS}

    label_column = {label: index for index, label in enumerate(EVAL_LABELS)}
    ordered = sorted(entry['annotations'], key=lambda a: (a['start_time'], a['end_time']))
    for annotation in ordered:
        column = label_column.get(annotation['label'])
        if column is None:  # not an evaluated label
            continue
        ann_totals[annotation['label']] += 1
        matched = (windows_match(starts, ends, annotation['start_time'], annotation['end_time'])
                   if n_windows else np.zeros(0, dtype=bool))
        n_ann[matched, column] += 1
        ann_windows[annotation['label']].append(np.flatnonzero(matched))

    entry['n_ann']       = n_ann
    entry['has_ann']     = n_ann > 0
    entry['ann_windows'] = ann_windows
    entry['ann_totals']  = ann_totals


def activate(logits, bias):
    """Sigmoid activation with the auricularia bias parameterisation."""
    transformed_bias = (bias - 1.0) * 10.0
    return 1.0 / (1.0 + np.exp(
        SIGMOID_SENSITIVITY * np.clip(logits + transformed_bias, -20, 20)
    ))


def count_window_level(entry, detected):
    """TP/FP/FN per label, one count per model window × label."""
    has_ann = entry['has_ann']
    tp = np.count_nonzero(detected & has_ann, axis=0)
    fp = np.count_nonzero(detected & ~has_ann, axis=0)
    # Every annotation overlapping a window with no detection is one FN, so a
    # long call missed by the model costs as many FNs as it spans windows.
    fn = np.where(detected, 0, entry['n_ann']).sum(axis=0)
    # An annotation that no window matched at all (e.g. it sits past the end of
    # the decoded audio) would otherwise vanish from the counts entirely.
    for column, label in enumerate(EVAL_LABELS):
        if entry['n_ann'][:, column].sum() == 0:
            fn[column] += entry['ann_totals'][label]
    return tp, fp, fn, np.zeros(len(EVAL_LABELS), dtype=np.int64)


def count_annotation_level(entry, detected, scores):
    """TP/FP/FN per label, one count per annotation event."""
    n_labels = len(EVAL_LABELS)
    tp = np.zeros(n_labels, dtype=np.int64)
    fn = np.zeros(n_labels, dtype=np.int64)

    for column, label in enumerate(EVAL_LABELS):
        claimed = set()
        for window_indices in entry['ann_windows'][label]:
            # Earliest annotation first (build_match_index sorted them), and each
            # detection can be claimed only once, so a burst of detections inside
            # one annotated call collapses into a single TP.
            unclaimed = [w for w in window_indices
                         if detected[w, column] and w not in claimed]
            if not unclaimed:
                fn[column] += 1
                continue
            best = max(unclaimed, key=lambda w: scores[w, column])
            claimed.add(best)
            tp[column] += 1

    # A detection overlapping an annotation of a different label is a genuine
    # confusion and still counts as a false positive.
    fp = np.count_nonzero(detected & ~entry['has_ann'], axis=0)
    return tp, fp, fn, np.zeros(n_labels, dtype=np.int64)


def count_file_level(entry, detected):
    """TP/FP/FN/TN per label, one count per recording: was it present at all?"""
    in_model = detected.any(axis=0)
    in_annotations = np.array([entry['ann_totals'][label] > 0 for label in EVAL_LABELS])
    tp = (in_annotations & in_model).astype(np.int64)
    fn = (in_annotations & ~in_model).astype(np.int64)
    fp = (~in_annotations & in_model).astype(np.int64)
    tn = (~in_annotations & ~in_model).astype(np.int64) if REPORT_TN \
        else np.zeros(len(EVAL_LABELS), dtype=np.int64)
    return tp, fp, fn, tn


def safe_divide(numerator, denominator):
    """Return None instead of raising, so an undefined metric stays an empty cell."""
    return None if denominator == 0 else numerator / denominator


def metrics_from_counts(tp, fp, fn):
    precision = safe_divide(tp, tp + fp)
    recall    = safe_divide(tp, tp + fn)
    if precision is None or recall is None or (precision + recall) == 0:
        f1 = None
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


print('Indexing annotations against model windows...')
for entry in validation_files:
    build_match_index(entry)

print(f'Sweeping {len(THRESHOLDS)} threshold(s) × {len(BIASES)} bias(es) '
      f'= {len(THRESHOLDS) * len(BIASES)} operating point(s)')
print('=' * 70)

metric_rows = []
sweep_start = time.time()

for bias in BIASES:
    # The model ran once; each bias is one pass of arithmetic over the stored logits.
    activated = [activate(entry['logits'], bias) for entry in validation_files]

    for threshold in THRESHOLDS:
        totals = {key: np.zeros(len(EVAL_LABELS), dtype=np.int64)
                  for key in ('TP', 'FP', 'FN', 'TN')}

        for entry, scores in zip(validation_files, activated):
            detected = scores >= threshold
            if GRANULARITY == 'annotation':
                tp, fp, fn, tn = count_annotation_level(entry, detected, scores)
            elif GRANULARITY == 'file':
                tp, fp, fn, tn = count_file_level(entry, detected)
            else:
                tp, fp, fn, tn = count_window_level(entry, detected)
            totals['TP'] += tp
            totals['FP'] += fp
            totals['FN'] += fn
            totals['TN'] += tn

        variant = f"{MODEL_NAME}__st{threshold:g}_sb{bias:g}"
        macro = {'precision': [], 'recall': [], 'f1': []}

        for column, label in enumerate(EVAL_LABELS):
            tp, fp, fn, tn = (int(totals[key][column]) for key in ('TP', 'FP', 'FN', 'TN'))
            precision, recall, f1 = metrics_from_counts(tp, fp, fn)
            # An undefined metric counts as 0 in the macro mean: a class the
            # model ignores should drag the average down, not be excluded.
            macro['precision'].append(precision or 0.0)
            macro['recall'].append(recall or 0.0)
            macro['f1'].append(f1 or 0.0)
            metric_rows.append({
                'variant': variant, 'score_threshold': threshold, 'sigmoid_bias': bias,
                'label': label, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                'support': tp + fn, 'precision': precision, 'recall': recall, 'f1': f1,
            })

        pooled = {key: int(totals[key].sum()) for key in ('TP', 'FP', 'FN', 'TN')}
        precision, recall, f1 = metrics_from_counts(pooled['TP'], pooled['FP'], pooled['FN'])
        metric_rows.append({
            'variant': variant, 'score_threshold': threshold, 'sigmoid_bias': bias,
            'label': MICRO_AVERAGE_LABEL, **pooled,
            'support': pooled['TP'] + pooled['FN'],
            'precision': precision, 'recall': recall, 'f1': f1,
        })
        metric_rows.append({
            'variant': variant, 'score_threshold': threshold, 'sigmoid_bias': bias,
            'label': MACRO_AVERAGE_LABEL, 'TP': None, 'FP': None, 'FN': None, 'TN': None,
            'support': len(EVAL_LABELS),
            **{key: sum(values) / len(values) for key, values in macro.items()},
        })

    print(f'  bias {bias:<5g} done  ({len(THRESHOLDS)} threshold(s))')

METRICS_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.validation_metrics.csv')
METRICS_COLUMNS = ['variant', 'score_threshold', 'sigmoid_bias', 'label',
                   'TP', 'FP', 'FN', 'TN', 'support', 'precision', 'recall', 'f1']
with open(METRICS_CSV_PATH, 'w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=METRICS_COLUMNS)
    writer.writeheader()
    for row in metric_rows:
        writer.writerow({key: ('' if row[key] is None else row[key]) for key in METRICS_COLUMNS})

print()
print('=' * 70)
print('Sweep complete.')
print(f'  Operating points : {len(THRESHOLDS) * len(BIASES)}')
print(f'  Metric rows      : {len(metric_rows)}')
print(f'  Time             : {time.time() - sweep_start:.1f}s')
print(f'  Saved to         : {METRICS_CSV_PATH}')

---
## Step 7 — Precision and recall curves

**This is the result of the notebook.** One figure per label. In each figure:

- The **x axis** is the score threshold
- **Solid lines with filled circles** are **precision** — of the detections the model made, how many
  were right
- **Dashed lines with hollow circles** are **recall** — of the calls that were really there, how
  many the model found
- **One colour per sigmoid bias**, named in the legend

Read them together: the threshold where the two lines of the same colour cross is roughly where
precision and recall are balanced (the F1 peak is usually near it). If precision is flat and low
everywhere, no threshold will save that class — the problem is the model or the label, not the
operating point.

The figures are saved as PNG files next to the metrics CSV on your Drive.

In [ ]:
#@title 📈 Plot precision and recall { display-mode: "form" }

#@markdown **Labels to plot** — semicolon-separated. Leave blank to plot every evaluated label.
LABELS_TO_PLOT = ""  #@param {type:"string"}

#@markdown **Also plot the averages** — adds figures for the micro average (counts pooled across
#@markdown labels, so common classes dominate) and the macro average (unweighted mean over labels,
#@markdown so rare classes count equally).
PLOT_AVERAGES = True  #@param {type:"boolean"}

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Distinct hues, assigned in a fixed order and never cycled. The bias values are
# ordered, but a one-hue light-to-dark ramp made neighbouring curves too close to
# tell apart, so identity wins over order here — the legend carries the order.
#
# This ordering was picked with a palette validator, and it is only safe up to
# FOUR curves. Curves cross, so any two colours can end up side by side; measured
# over every pair on this surface:
#   4 hues  worst pair ΔE 9.2 colour-blind / 16.3 normal vision  — safe
#   5 hues  magenta↔orange 12.9 normal vision                    — below the 15 floor
#   6 hues  green↔orange 3.2 under protanopia                    — near-identical
#   8 hues  red↔orange 7.1 normal vision                         — visibly similar
# Past four there is no colour-only fix: any further hue collides with one already
# in use. The cell warns instead of pretending otherwise.
_HUES = ['#2a78d6',   # blue
         '#eb6834',   # orange
         '#1baf7a',   # aqua
         '#4a3aa7',   # violet
         '#e87ba4',   # magenta
         '#008300',   # green
         '#eda100',   # yellow
         '#e34948']   # red
# Fallback for a sweep with more biases than there are hues: a single-hue ramp,
# light to dark. Curves stop being individually identifiable at that point — the
# cell says so rather than inventing a ninth hue.
_RAMP    = ['#86b6ef', '#5598e7', '#3987e5', '#256abf', '#184f95', '#0d366b']
_INK     = '#0b0b0b'
_MUTED   = '#898781'
_GRID    = '#e1e0d9'
_SURFACE = '#fcfcfb'
_AXIS    = '#52514e'


SAFE_BIAS_COLOURS = 4


def bias_colors(count):
    """One colour per bias curve, taken in fixed order from the palette above."""
    if count <= len(_HUES):
        if count > SAFE_BIAS_COLOURS:
            print(f'NOTE: {count} sigmoid biases means {count} curve pairs per figure. Only the')
            print(f'      first {SAFE_BIAS_COLOURS} colours are guaranteed to be distinguishable '
                  f'where curves cross')
            print('      (and the 6th onwards are near-identical to red-green colour blindness).')
            print('      Prefer 4 or fewer biases per run, or read the exact numbers in the')
            print('      metrics CSV instead of off the figure.')
        return _HUES[:count]

    print(f'NOTE: {count} sigmoid biases is more than the {len(_HUES)} distinct colours')
    print('      available, so the curves fall back to one hue from light to dark and')
    print('      become hard to tell apart. Consider sweeping fewer biases.')
    anchors = [tuple(int(step[i:i + 2], 16) for i in (1, 3, 5)) for step in _RAMP]
    colors = []
    for index in range(count):
        position = index / (count - 1) * (len(anchors) - 1)
        low      = min(int(position), len(anchors) - 2)
        fraction = position - low
        colors.append('#%02x%02x%02x' % tuple(
            round(anchors[low][c] + fraction * (anchors[low + 1][c] - anchors[low][c]))
            for c in range(3)
        ))
    return colors


BIAS_COLORS = dict(zip(BIASES, bias_colors(len(BIASES))))


def style_axis(ax, title, subtitle):
    ax.set_xlabel('score threshold', color=_AXIS, fontsize=10)
    ax.set_ylabel('precision  |  recall', color=_AXIS, fontsize=10)
    ax.set_title(title, color=_INK, fontsize=13, loc='left', pad=30)
    ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=_MUTED, fontsize=9, va='bottom')
    ax.set_ylim(-0.02, 1.02)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.grid(axis='y', color=_GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(colors=_MUTED, labelsize=9)
    for side, spine in ax.spines.items():
        spine.set_visible(side == 'bottom')
        spine.set_color('#c3c2b7')


def plot_label(label, rows, output_path):
    """Draw one label's precision/recall curves, one coloured pair per bias."""
    fig, ax = plt.subplots(figsize=(7.5, 5), dpi=120)
    fig.patch.set_facecolor(_SURFACE)
    ax.set_facecolor(_SURFACE)

    drawn = 0
    for bias in BIASES:
        points = sorted((r for r in rows if r['sigmoid_bias'] == bias),
                        key=lambda r: r['score_threshold'])
        # An undefined metric is a real result for a threshold that suppressed the
        # class, but not a plottable one — those points drop out of the curve.
        precision_points = [(r['score_threshold'], r['precision']) for r in points
                            if r['precision'] is not None]
        recall_points    = [(r['score_threshold'], r['recall']) for r in points
                            if r['recall'] is not None]
        color = BIAS_COLORS[bias]
        if precision_points:
            ax.plot(*zip(*precision_points), color=color, linewidth=1.8,
                    marker='o', markersize=6.5, markeredgecolor=_SURFACE,
                    markeredgewidth=1.2, zorder=3)
            drawn += 1
        if recall_points:
            ax.plot(*zip(*recall_points), color=color, linewidth=1.8, linestyle='--',
                    marker='o', markersize=6.5, markerfacecolor=_SURFACE,
                    markeredgecolor=color, markeredgewidth=1.6, zorder=3)
            drawn += 1

    if not drawn:
        plt.close(fig)
        return None

    support = max((r['support'] or 0) for r in rows)
    style_axis(ax, label, f'{support} annotation(s) · {GRANULARITY}-level · {MODEL_NAME}')

    handles = [Line2D([], [], color=BIAS_COLORS[b], linewidth=2.4, label=f'bias {b:g}')
               for b in BIASES]
    handles += [
        Line2D([], [], color=_MUTED, linewidth=1.8, marker='o', markersize=6.5,
               markeredgecolor=_SURFACE, label='precision'),
        Line2D([], [], color=_MUTED, linewidth=1.8, linestyle='--', marker='o', markersize=6.5,
               markerfacecolor=_SURFACE, markeredgecolor=_MUTED, label='recall'),
    ]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1),
              frameon=False, fontsize=9, labelcolor=_INK)

    fig.tight_layout()
    fig.savefig(output_path, facecolor=_SURFACE, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return output_path


def safe_filename(label):
    return ''.join(c if c.isalnum() or c in '._-' else '-' for c in label)


requested = parse_list(LABELS_TO_PLOT) or list(EVAL_LABELS)
unknown   = [label for label in requested if label not in EVAL_LABELS]
if unknown:
    raise ValueError(f"Label(s) not evaluated: {unknown}\nAvailable: {EVAL_LABELS}")
if PLOT_AVERAGES:
    requested = requested + [MICRO_AVERAGE_LABEL, MACRO_AVERAGE_LABEL]

FIGURES_DIR = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

written = []
for label in requested:
    rows = [r for r in metric_rows if r['label'] == label]
    path = plot_label(label, rows,
                      os.path.join(FIGURES_DIR, f'{MODEL_NAME}_{safe_filename(label)}.png'))
    if path:
        written.append(path)
    else:
        print(f"'{label}': no plottable point — precision and recall are undefined at every "
              f"threshold (the model never predicted it, or it was never annotated).")

print()
print(f'{len(written)} figure(s) saved to: {FIGURES_DIR}')

---
## Step 8 — Best operating point per label

A curve shows the whole trade-off; this table picks one point off it. For each label it reports the
threshold and bias with the **highest F1 score** — the best balance of precision and recall.

Treat it as a starting suggestion, not a verdict. If your project cares more about not missing
calls, pick a lower threshold from the curve and accept the precision cost; if it cares more about
not reviewing false positives, pick a higher one.

In [ ]:
#@title 🏆 Best operating point { display-mode: "form" }

import pandas as pd

best_rows = []
for label in list(EVAL_LABELS) + [MICRO_AVERAGE_LABEL, MACRO_AVERAGE_LABEL]:
    scored = [r for r in metric_rows if r['label'] == label and r['f1'] is not None]
    if not scored:
        continue
    # Ties go to the lower threshold: the same F1 for less aggressive filtering.
    best = min(scored, key=lambda r: (-r['f1'], r['score_threshold']))
    best_rows.append({
        'label': label,
        'score_threshold': best['score_threshold'],
        'sigmoid_bias': best['sigmoid_bias'],
        'precision': None if best['precision'] is None else round(best['precision'], 3),
        'recall': None if best['recall'] is None else round(best['recall'], 3),
        'f1': round(best['f1'], 3),
        'TP': best['TP'], 'FP': best['FP'], 'FN': best['FN'],
        'support': best['support'],
    })

BEST_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.best_operating_points.csv')
best_table = pd.DataFrame(best_rows)
best_table.to_csv(BEST_CSV_PATH, index=False)

print(f'Model       : {MODEL_NAME}')
print(f'Granularity : {GRANULARITY}  |  Match: {MATCH_MODE}')
print(f'Recordings  : {len(validation_files)}')
print()
display(best_table)
print()
print(f'Metrics for every operating point : {METRICS_CSV_PATH}')
print(f'Best operating points             : {BEST_CSV_PATH}')
print(f'Figures                           : {FIGURES_DIR}')

---
### Done

You now have, on your Google Drive:

- `<model>.validation_metrics.csv` — every label at every operating point (TP/FP/FN, precision,
  recall, F1)
- `<model>.best_operating_points.csv` — the highest-F1 threshold and bias per label
- `<model>_figures/` — one precision/recall figure per label

**To try more operating points**, edit the *Sweep & Validation* form in Step 2 and re-run from
Step 6. With the logits cache enabled, Step 5 will not re-run the model.

---

Created by [biodiversica](https://biodiversica.xyz). For issues, questions, or feedback, open an
issue on [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) or visit
[biodiversica.xyz](https://biodiversica.xyz).